In [1]:
import os
import dill
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold


from sklearn.metrics import (auc,roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, brier_score_loss,
                             roc_curve, precision_recall_curve)
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

import shap

In [2]:
all_IV = pd.read_csv('D:/2024BI_Journal/2025data/IV_mdro_basic_37180.csv')
print(all_IV.shape)
all_IV.MDR_LABEL.value_counts()

(37180, 52)


0    33707
1     3473
Name: MDR_LABEL, dtype: int64

In [3]:
missing_cols = all_IV.columns[all_IV.isnull().any()]
print(missing_cols)

Index(['DOD', 'DEATHTIME'], dtype='object')


In [4]:
all_IV.columns

Index(['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ADMITTIME', 'INTIME',
       'DISCHTIME', 'OUTTIME', 'GENDER', 'DOB', 'DOD', 'DEATHTIME',
       'ADMISSION_TYPE', 'ADMISSION_LOCATION', 'RACE', 'FIRST_CAREUNIT',
       'LAST_CAREUNIT', 'LOS', 'AGE', 'ANCHOR_YEAR_GROUP', 'FIRST_HADM',
       'FIRST_ICU', 'DIEINHOSPITAL', 'DIEINICU', 'Readmission_30',
       'Readmission_60', 'HOURS_FROM_ADMIT', 'ICU_within_12hr_of_admit',
       'Multiple_ICUs', 'ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE',
       'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS',
       'MDR_LABEL', 'MDR_CHECKED', 'AB_ID_COUNT', 'AB_ID_FILTERED_COUNT',
       'ATE', 'ATE_FILTERED', 'MDR_BEFORE', 'BLOOD CULTURE', 'URINE', 'SWAB',
       'SPUTUM', 'MRSA SCREEN', 'BAL', 'TISSUE', 'PERITONEAL FLUID',
       'PLEURAL FLUID', 'ABSCESS', 'CSF', 'JOINT FLUID'],
      dtype='object')

In [5]:
old_IV = pd.read_csv('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/All_40815_df_minmax_24.csv')
old_IV = old_IV[old_IV.first_icu_stay == 1]
print(old_IV.shape)

(37180, 250)


In [6]:
old_IV = old_IV[['ICUSTAY_ID','Age', 'gender','admission_type','first_hosp_stay', 'first_careunit','admin_counts','A40', 'D64', 'D69', 'E11', 'E66', 'E78', 'E8497', 'E87', 'E8798', 'E89', 'F05', 'F10', 'F32', 'F41', 'G47', 'I10', 'I12', 'I21', 'I25', 'I27', 'I34', 'I47', 'I50', 'I95', 'J18', 'J44', 'J69', 'J98', 'K22', 'N17', 'N18', 'N39', 'R00', 'R09', 'R68', 'R78', 'T78', 'T81', 'Z51', 'Z91', 'Z95', 'A41', 'CHART_220045', 'CHART_220179', 'CHART_220180', 'CHART_220181', 'CHART_220210', 'CHART_220277', 'CHART_223834', 'CHART_226253', 'CHART_224639', 'CHART_220224', 'CHART_220228', 'CHART_220235', 'CHART_223830', 'CHART_225667', 'CHART_225668', 'CHART_220545', 'CHART_220546', 'CHART_220602', 'CHART_220615', 'CHART_220645', 'CHART_225624', 'CHART_225625', 'CHART_227073', 'CHART_227443', 'CHART_227457', 'CHART_227465', 'CHART_227466', 'CHART_227467', 'CHART_227442', 'CHART_220587', 'CHART_220621', 'CHART_220644', 'CHART_225612', 'CHART_225690', 'CHART_220227', 'CHART_220274', 'CHART_227429', 'CHART_220603', 'CHART_220632', 'CHART_229357', 'CHART_229358', 'CHART_229359', 'CHART_229360', 'CHART_229361', 'CHART_220050', 'CHART_220051', 'CHART_220052', 'CHART_220339', 'CHART_224690', 'CHART_224697', 'CHART_224700', 'CHART_220074', 'CHART_227456', 'CHART_226534', 'CHART_226536', 'CHART_226537', 'CHART_226540', 'CHART_227464', 'CHART_225651', 'CHART_224144', 'CHART_223762', 'CHART_225638', 'CHART_229355', 'CHART_Cate_225092', 'CHART_Cate_225113', 'CHART_Cate_225103', 'CHART_Cate_225091', 'CHART_Cate_225112', 'CHART_Cate_225118', 'CHART_Cate_225126', 'CHART_Cate_225094', 'CHART_Cate_225124', 'CHART_Cate_225108', 'CHART_Str_220739', 'CHART_Str_223900', 'CHART_Str_223901', 'CHART_Str_225101', 'CHART_Str_228699', 'LAB_50802', 'LAB_51249', 'LAB_51277', 'LAB_51279', 'LAB_52172', 'LAB_51516', 'LAB_50811', 'LAB_51493', 'LAB_51300', 'LAB_Str_51463', 'PROC_229351', 'PROC_225752', 'PROC_224264', 'PROC_225792', 'PROC_224270', 'PROC_225794', 'PROC_225802', 'PROC_225441', 'PROC_224268', 'PROC_224272', 'PROC_225805', 'PROC_224273', 'PROC_Cate_225402', 'PROC_Cate_225401', 'PROC_Cate_221214', 'PROC_Cate_221217', 'PROC_Cate_227194', 'PROC_Cate_224385', 'PROC_Cate_221216', 'PROC_Cate_227712', 'PROC_Cate_225454', 'PROC_Cate_225966', 'PROC_Cate_225444', 'PROC_Cate_225440', 'PROC_Cate_225439', 'PROC_Cate_223253', 'PROC_Cate_225427', 'PROC_Cate_225400', 'PROC_Cate_225451', 'PROC_Cate_225817', 'PROC_Cate_221223', 'PROC_Cate_225816', 'PROC_Cate_225814', 'PROC_Cate_225445', 'PROC_Cate_225466', 'PROC_Cate_225464', 'PROC_Cate_225399', 'PROC_Cate_225430', 'PROC_Cate_225434', 'PROC_Cate_225448', 'PROC_Cate_228715', 'PROC_Cate_225433', 'PROC_Cate_225479', 'PROC_Cate_225437', 'PROC_Cate_225475', 'PROC_Cate_227713', 'PROC_Cate_225467', 'PROC_Cate_225449', 'PROC_Cate_225472', 'PROC_Cate_225450', 'PROC_Cate_225967', 'PROC_Cate_225442', 'MED_221749', 'MED_221744', 'MED_222168', 'MED_225168', 'MED_223258', 'MED_221906', 'MED_225154', 'MED_221468', 'MED_222042', 'MED_221794', 'MED_222315', 'MED_222056', 'MED_225152', 'MED_221662', 'MED_225170', 'MED_221289', 'MED_221986', 'MED_221555', 'MED_221653', 'MED_221319', 'MED_229630', 'MED_221282', 'MED_229617']]

In [7]:
all_IV = pd.merge(all_IV,old_IV,on='ICUSTAY_ID',how='left')

In [8]:
missing_cols = all_IV.columns[all_IV.isnull().any()]
print(missing_cols)

Index(['DOD', 'DEATHTIME'], dtype='object')


In [ ]:
ventilation = pd.read_csv('D:/2025UTI/Data/MIMIC4_code/ventilation.csv')
ventilation.columns = ['ICUSTAY_ID', 'starttime', 'endtime', 'ventilation_status']
ventilation = pd.merge(ventilation,all_IV[['ICUSTAY_ID','ADMITTIME','INTIME', 'OUTTIME']],
                       on='ICUSTAY_ID',how='left')
ventilation = ventilation.dropna(subset=['ADMITTIME'])
datetime_cols = ['ADMITTIME', 'INTIME','OUTTIME', 'starttime']
ventilation[datetime_cols] = ventilation[datetime_cols].apply(pd.to_datetime)

ventilation_icu_window_criteria = ventilation['starttime'].between(
    ventilation['ADMITTIME'],
    ventilation['INTIME'] + pd.DateOffset(1))
ventilation_icu = ventilation[ventilation_icu_window_criteria]
InvasiveVent = ventilation_icu[ventilation_icu.ventilation_status == 'InvasiveVent']
InvasiveVent_ids = list(InvasiveVent.ICUSTAY_ID.unique())
InvasiveVent_ids = [int(i) for i in InvasiveVent_ids]
all_IV['InvasiveVent'] = all_IV['ICUSTAY_ID'].isin(InvasiveVent_ids).astype(int)

In [152]:
print(all_IV.columns.tolist())

['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ADMITTIME', 'INTIME', 'DISCHTIME', 'OUTTIME', 'GENDER', 'DOB', 'DOD', 'DEATHTIME', 'ADMISSION_TYPE', 'ADMISSION_LOCATION', 'RACE', 'FIRST_CAREUNIT', 'LAST_CAREUNIT', 'LOS', 'AGE', 'ANCHOR_YEAR_GROUP', 'FIRST_HADM', 'FIRST_ICU', 'DIEINHOSPITAL', 'DIEINICU', 'Readmission_30', 'Readmission_60', 'HOURS_FROM_ADMIT', 'ICU_within_12hr_of_admit', 'Multiple_ICUs', 'ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL', 'MDR_CHECKED', 'AB_ID_COUNT', 'AB_ID_FILTERED_COUNT', 'ATE', 'ATE_FILTERED', 'MDR_BEFORE', 'BLOOD CULTURE', 'URINE', 'SWAB', 'SPUTUM', 'MRSA SCREEN', 'BAL', 'TISSUE', 'PERITONEAL FLUID', 'PLEURAL FLUID', 'ABSCESS', 'CSF', 'JOINT FLUID', 'Age', 'gender', 'admission_type', 'first_hosp_stay', 'first_careunit', 'admin_counts', 'A40', 'D64', 'D69', 'E11', 'E66', 'E78', 'E8497', 'E87', 'E8798', 'E89', 'F05', 'F10', 'F32', 'F41', 'G47', 'I10', 'I12', 'I21', 'I25', 'I27', 'I3

## 前期准备

In [10]:
id_mapping = {
    "CHART_225612": "Alkaline Phosphate",
    "CHART_224700": "Total PEEP Level",
    "CHART_227457": "Platelet Count",
    "CHART_225625": "Calcium non-ionized",
    "LAB_51249": "MCHC",
    "CHART_220181": "Non-invasive Blood Pressure mean",
    "CHART_226537": "Glucose (whole blood)",
    "LAB_51516": "WBC",
    "CHART_225651": "Direct Bilirubin",
    "CHART_227073": "Anion gap",
    "MED_221906": "Norepinephrine",
    "CHART_225624": "BUN",
    "CHART_225690": "Total Bilirubin",
    "CHART_220545": "Hematocrit (serum)",
    "CHART_220228": "Hemoglobin",
    "CHART_220277": "O2 saturation pulse",
    "CHART_220210": "Respiratory Rate",
    "CHART_225668": "Lactic Acid",
    "CHART_Str_225101": "Use of assistive devices",
    "CHART_227466": "PTT",
    "LAB_51279": "Red Blood Cells",
    "CHART_220235": "Arterial CO2 Pressure",
    "LAB_51277": "RDW",
    "CHART_220179": "Non-invasive Blood Pressure systolic",
    "CHART_220045": "Heart Rate",
    "CHART_220615": "Creatinine (serum)",
    "CHART_227465": "Prothrombin time",
    "CHART_227443": "HCO3 (serum)",
    "PROC_224270": "Dialysis Catheter",
    'PROC_225802':'Dialysis - CRRT',
    'CHART_223762':'Temperature',
    'PROC_225792':'Invasive Ventilation',
    "PROC_225752": "Arterial Line",
    'CHART_Cate_225113':'Experiencing pain',
     'CHART_Cate_225092':'Self ADL',
    #'T78': 'Adverse effects',
    'N18': 'CKD',#Chronic kidney disease (CKD)
    'J18': 'Pneumonia',
    #'I95': 'Hypotension',
    'E11': 'Type 2 Diabetes',#Type 2 diabetes mellitus
    'J98': 'Respiratory Disorders',
    'D64': 'Anemias',
    #'F10': 'Alcohol',
    'A40': 'Streptococcal sepsis',
    'J44': 'COPD ',#chronic obstructive pulmonary disease
    'N39': 'Urinary Disorders',#Other disorders of urinary system
    'gender':'Gender',
    'ab_id_count':'Antibiotics use',
    'ATE':'AET',
    'admission_type':'Admission type',
    'first_hosp_stay':'First hosp stay', 
    'first_careunit':'First careunit',
    'admin_counts':'Hosp counts', 
    'MDR_before':'MDROs before',
    'BAL':'BAL Culture', 'CSF':'CSF Culture', 
    'ABSCESS':'Abscess Culture', 'PERITONEAL FLUID':'Peritoneal Fluid Culture', 
    'SPUTUM':'Sputum Culture',  'URINE':'Urine Culture', 
    'MRSA SCREEN':'MRSA Screen','BLOOD CULTURE':'Blood Culture',
    'SWAB':'Swab',
    
    "PROC_225794": "Non-invasive Ventilation",
    "MED_221468": "Diltiazem",
    "LAB_50802": "Base Excess",
    "MED_221289": "Epinephrine",
    "CHART_224144": "Blood Flow (ml/min)",
    "PROC_224273": "Presep Catheter",
    "CHART_220644": "ALT",
    "MED_221986": "Milrinone",
    "PROC_Cate_225816": "Wound Culture",
    "CHART_224639": "Daily Weight",
    "MED_223258": "Insulin - Regular",
    "CHART_226253": "SpO2 Desat Limit",
    "CHART_229357": "Absolute Count - Neuts",
    "PROC_Cate_225444": "Pan Culture",
    "CHART_Str_223901": "GCS - Motor Response",
    "MED_222168": "Propofol",
    "CHART_229360": "Absolute Count - Eos",
    #"PROC_Cate_225451": "Sputum Culture",
    #"PROC_Cate_225445": "Paracentesis",
    "MED_225170": "Platelets",
    "MED_225168": "Packed Red Blood Cells",
    "CHART_220602": "Chloride (serum)",
    "MED_221319": "Alteplase (TPA)",
    "PROC_225805": "Peritoneal Dialysis",
    "MED_229617": "Epinephrine.",
    #"PROC_229351": "Foley Catheter",
    "PROC_224268": "Trauma line",
    "MED_229630": "Phenylephrine (50/250)",
    "PROC_Cate_225479": "Thoracentesis",
    "CHART_220587": "AST",
    "PROC_Cate_225400": "Bronchoscopy",
    "PROC_224272": "IABP line",
    "CHART_220180": "Non Invasive Blood Pressure diastolic",
    "CHART_225667": "Ionized Calcium",
    "CHART_227467": "INR",
    "CHART_226534": "Sodium (whole blood)",
    "MED_222042": "Nicardipine",
    "CHART_229359": "Absolute Count - Monos",
    "CHART_227442": "Potassium (serum)",
    "PROC_Cate_225399": "Lumbar Puncture",
    "CHART_223830": "PH (Arterial)",
    #"PROC_Cate_225401": "Blood Cultured",
    "CHART_220052": "Arterial Blood Pressure mean",
    "CHART_227456": "Albumin",
    "MED_221662": "Dopamine",
    "CHART_220274": "PH (Venous)",
    "CHART_223834": "O2 Flow",
    "CHART_220051": "Arterial Blood Pressure diastolic",
    "MED_221282": "Adenosine",
    "LAB_52172": "RDW-SD",
    "MED_221794": "Furosemide (Lasix)",
    "CHART_227429": "Troponin-T",
    "CHART_220645": "Sodium (serum)",
    "CHART_220339": "PEEP set",
    "LAB_51493": "RBC",
    "CHART_220050": "Arterial Blood Pressure systolic",
    "LAB_Str_51463": "Bacteria",
    "CHART_220621": "Glucose (serum)",
    "MED_225154": "Morphine Sulfate",
    "CHART_224690": "Respiratory Rate (Total)",
    "PROC_225441": "Hemodialysis",
    "MED_225152": "Heparin Sodium",
    "MED_222056": "Nitroglycerin",
    "CHART_229355": "Absolute Neutrophil Count",
    "CHART_229361": "Absolute Count - Basos",
    "LAB_51300": "WBC Count",
    "PROC_Cate_225454": "Urine Culture",
    "CHART_220227": "Arterial O2 Saturation",
    "CHART_226540": "Hematocrit (whole blood - calc)",
    "CHART_227464": "Potassium (whole blood)",
    "PROC_Cate_225817": "BAL Fluid Culture",
    "CHART_220603": "Cholesterol",
    "CHART_220074": "Central Venous Pressure",
    "CHART_224697": "Mean Airway Pressure",
    "MED_221653": "Dobutamine",
    "MED_222315": "Vasopressin",
    "CHART_220632": "LDH",
    "MED_221749": "Phenylephrine",
    "MED_221555": "Cisatracurium",
    #'CHART_Cate_225118':'Difficulty swallowing',
    'CHART_226536':'Chloride (whole blood)',
#     'CHART_Cate_225094':'History of slips / falls',
    'CHART_220224':'PO2 (Arterial)',
    'CHART_Cate_225124':'Unintentional weight loss >10 lbs.',
    'CHART_225638':'Differential-Bands',
    'MED_221744':'Fentanyl',
    'PROC_Cate_224385':'Intubation',
    'PROC_224264':'PICC Line'
#     'CHART_Cate_225108':'Tobacco use'
}

In [11]:
models = [
    ('logistic_regression', {
#         'solver':'liblinear','class_weight': 'balanced',
# 'penalty': 'l1','random_state':12345
    }),
    ('random_forest', {
#         'criterion':'entropy',
# 'random_state':12345,'n_estimators': 150,
# 'min_samples_split': 10, 'min_samples_leaf':
# 2, 'max_features': 3, 'max_depth': 15,
# 'class_weight': {0: 0.000181422351233672, 1: 0.0005906674542232723}, 'bootstrap':
# False
                      }),

        ('xgboost', {
#             'n_estimators':100, 'min_child_weight':2,
# 'gamma':0, 'subsample':0.8,
# 'colsample_bytree':0.8,
# 'objective':'binary:logistic', 'n_jobs':-1,
# 'seed':27,'scale_pos_weight': 1, 'max_depth':
# 3, 'learning_rate': 0.1,
                     'eval_metric': 'mlogloss'}),
         ]

In [12]:
C_P_M_id = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/d_items.csv.gz')
D_id = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/d_icd_diagnoses.csv.gz')
L_id = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/d_labitems.csv.gz')
diag_col = ['A40', 'D64', 'D69', 'E11', 'E66', 'E78', 'E8497', 'E87', 'E8798', 'E89', 'F05', 'F10', 'F32', 'F41', 'G47', 'I10', 'I12', 'I21', 'I25', 'I27', 'I34', 'I47', 'I50', 'I95', 'J18', 'J44', 'J69', 'J98', 'K22', 'N17', 'N18', 'N39', 'R00', 'R09', 'R68', 'R78', 'T78', 'T81', 'Z51', 'Z91', 'Z95', 'A41']

In [170]:
all_IV.rename(columns=id_mapping)[['ICUSTAY_ID', 'FIRST_CAREUNIT','MDR_BEFORE','MRSA Screen','ATE_FILTERED','Gender','RACE','AGE','ADMISSION_TYPE','Use of assistive devices','InvasiveVent']].to_csv('D:/2024BI_Journal/2025data/HeterogeneousSubset.csv',index=False)

In [153]:
IV_run = all_IV.copy()
labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']

#'icu_counts','first_icu_stay','ab_id_filtered_count','ATE_filtered', 
f_b = ['Age', 'gender','admission_type','first_hosp_stay', 'first_careunit','admin_counts',
      'AB_ID_COUNT','ATE_FILTERED', 'MDR_BEFORE']

f_d = ['CHART_Cate_225092','CHART_Cate_225113','N18', 'J18',  'E11', 'J98', 'D64','A40', 'J44','N39']

f_cul = ['BAL', 'CSF', 'ABSCESS', 'PERITONEAL FLUID', 'SPUTUM',  'URINE', 'MRSA SCREEN','BLOOD CULTURE', 'SWAB']
#IV_run[f_cul] = IV_run[f_cul].apply(lambda x: (x != 0).astype(int))

ft_s = ['CHART_220545', 'LAB_51249', 'LAB_51279', 'CHART_225624', 
        'CHART_220228',  'CHART_223762', 'CHART_225612', 'CHART_220615', 'LAB_51277', 
        'CHART_220179', 'CHART_Str_225101', 'LAB_51516', 'CHART_220235', 'CHART_225651', 
        'CHART_226537', 'CHART_227073', 'CHART_220277', 'CHART_220045', 'CHART_225690', 'MED_221906', 
        'CHART_225625', 'CHART_227443', 'CHART_227465', 'CHART_220210',  'CHART_227466', 
        'CHART_224700', 'CHART_227457', 'CHART_225668',
        'PROC_225802', 'PROC_224270', 'PROC_225792','PROC_225752']
len(ft_s)

32

In [154]:
features = ft_s + f_b + f_d + f_cul
print(len(features))

60


In [155]:
all_df = IV_run[features+labels + ['RACE','AGE','ADMISSION_TYPE','FIRST_CAREUNIT','InvasiveVent']].rename(columns=id_mapping)

In [156]:
print(all_df.columns.tolist())

['Hematocrit (serum)', 'MCHC', 'Red Blood Cells', 'BUN', 'Hemoglobin', 'Temperature', 'Alkaline Phosphate', 'Creatinine (serum)', 'RDW', 'Non-invasive Blood Pressure systolic', 'Use of assistive devices', 'WBC', 'Arterial CO2 Pressure', 'Direct Bilirubin', 'Glucose (whole blood)', 'Anion gap', 'O2 saturation pulse', 'Heart Rate', 'Total Bilirubin', 'Norepinephrine', 'Calcium non-ionized', 'HCO3 (serum)', 'Prothrombin time', 'Respiratory Rate', 'PTT', 'Total PEEP Level', 'Platelet Count', 'Lactic Acid', 'Dialysis - CRRT', 'Dialysis Catheter', 'Invasive Ventilation', 'Arterial Line', 'Age', 'Gender', 'Admission type', 'First hosp stay', 'First careunit', 'Hosp counts', 'AB_ID_COUNT', 'ATE_FILTERED', 'MDR_BEFORE', 'Self ADL', 'Experiencing pain', 'CKD', 'Pneumonia', 'Type 2 Diabetes', 'Respiratory Disorders', 'Anemias', 'Streptococcal sepsis', 'COPD ', 'Urinary Disorders', 'BAL Culture', 'CSF Culture', 'Abscess Culture', 'Peritoneal Fluid Culture', 'Sputum Culture', 'Urine Culture', 'MRSA

In [157]:
features = list(set(all_df.columns.tolist())-set(labels+['RACE','AGE','ADMISSION_TYPE','FIRST_CAREUNIT','InvasiveVent']))
len(features)

60

In [158]:
print(features)

['Peritoneal Fluid Culture', 'HCO3 (serum)', 'Dialysis - CRRT', 'Respiratory Disorders', 'Streptococcal sepsis', 'Direct Bilirubin', 'PTT', 'Temperature', 'Norepinephrine', 'RDW', 'Invasive Ventilation', 'Red Blood Cells', 'Type 2 Diabetes', 'Hemoglobin', 'Total PEEP Level', 'Use of assistive devices', 'Non-invasive Blood Pressure systolic', 'Glucose (whole blood)', 'Pneumonia', 'First careunit', 'Abscess Culture', 'MRSA Screen', 'Respiratory Rate', 'Anemias', 'Hematocrit (serum)', 'Arterial Line', 'Self ADL', 'Calcium non-ionized', 'Arterial CO2 Pressure', 'O2 saturation pulse', 'Age', 'Hosp counts', 'Swab', 'CKD', 'Creatinine (serum)', 'BUN', 'Admission type', 'Urinary Disorders', 'Urine Culture', 'AB_ID_COUNT', 'Experiencing pain', 'COPD ', 'ATE_FILTERED', 'Dialysis Catheter', 'CSF Culture', 'Prothrombin time', 'Blood Culture', 'Sputum Culture', 'Lactic Acid', 'WBC', 'Platelet Count', 'Alkaline Phosphate', 'Total Bilirubin', 'Heart Rate', 'Gender', 'MCHC', 'BAL Culture', 'First hosp

In [159]:
cultures = ['BAL Culture', 'CSF Culture', 'Abscess Culture', 'Peritoneal Fluid Culture', 'Sputum Culture', 'Urine Culture', 'MRSA Screen', 'Blood Culture', 'Swab']

In [29]:
import baseM

## 子组预测

### Based on MDRO History

In [38]:
print(Counter(all_df['MDR_BEFORE']))
IV_MDR_before = all_df[all_df['MDR_BEFORE']>0]
IV_MDR_before.shape

Counter({0: 36536, 1: 644})


(644, 70)

In [39]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = IV_MDR_before.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({1: 388, 0: 256}) (644, 70) 60.2%
Train: Counter({1: 264, 0: 186})
Test: Counter({1: 124, 0: 70})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:08<00:00,  2.46it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7636 (0.6803-0.8468),0.8624 (0.8139-0.9108),0.6959 (0.6441-0.7477),0.7242 (0.6727-0.7757),0.6959 (0.6441-0.7477),0.7016 (0.6522-0.751),0.3889 (0.2716-0.5062),0.2074 (0.162-0.2528)
RF,0.7881 (0.7046-0.8716),0.8561 (0.7797-0.9324),0.7138 (0.6436-0.784),0.7057 (0.6274-0.784),0.7138 (0.6436-0.784),0.704 (0.624-0.784),0.3335 (0.169-0.4981),0.1904 (0.1714-0.2093)
XGB,0.7615 (0.6803-0.8427),0.858 (0.8074-0.9086),0.7138 (0.649-0.7786),0.7185 (0.6494-0.7876),0.7138 (0.649-0.7786),0.7167 (0.6522-0.7813),0.3683 (0.206-0.5307),0.2029 (0.1617-0.2441)


In [44]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['Arterial CO2 Pressure', 'Use of assistive devices',
       'Urinary Disorders', 'WBC', 'Platelet Count', 'First careunit',
       'Lactic Acid', 'Total Bilirubin', 'Arterial Line',
       'Calcium non-ionized'], dtype=object)

### Based on Testing History

In [45]:
print(Counter(all_df['MRSA Screen']))
IV_MRSA = all_df[all_df['MRSA Screen']>0]
IV_MRSA.shape

Counter({1: 23075, 0: 10092, 10: 2997, 2: 871, 7: 90, 3: 19, 8: 15, 6: 12, 5: 6, 4: 2, 9: 1})


(27088, 70)

In [46]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = IV_MRSA.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 24570, 1: 2518}) (27088, 70) 9.3%
Train: Counter({0: 17243, 1: 1718})
Test: Counter({0: 7327, 1: 800})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:15<00:00,  1.30it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7763 (0.7636-0.7889),0.3924 (0.3683-0.4166),0.9101 (0.9063-0.9139),0.8935 (0.8863-0.9006),0.9101 (0.9063-0.9139),0.8819 (0.8759-0.8878),0.2944 (0.2601-0.3287),0.074 (0.0714-0.0765)
RF,0.7793 (0.7598-0.7987),0.3449 (0.3051-0.3846),0.905 (0.8978-0.9121),0.8951 (0.8763-0.914),0.905 (0.8978-0.9121),0.863 (0.8534-0.8726),0.1212 (0.0603-0.1821),0.0764 (0.0719-0.081)
XGB,0.7503 (0.7288-0.7718),0.3348 (0.3019-0.3677),0.9069 (0.9021-0.9117),0.8855 (0.8784-0.8926),0.9069 (0.9021-0.9117),0.8806 (0.8735-0.8876),0.275 (0.2457-0.3043),0.08 (0.0758-0.0841)


In [47]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Streptococcal sepsis', 'Self ADL',
       'Urinary Disorders', 'COPD ', 'First careunit', 'Hosp counts',
       'First hosp stay', 'Use of assistive devices'], dtype=object)

### Based on Antibiotic Use

In [48]:
print(Counter(all_df['ATE_FILTERED']))
IV_ATE_filtered = all_df[all_df['ATE_FILTERED']>0]
IV_ATE_filtered.shape

Counter({1: 28120, 0: 9060})


(28120, 70)

In [49]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = IV_ATE_filtered.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 25177, 1: 2943}) (28120, 70) 10.5%
Train: Counter({0: 17618, 1: 2066})
Test: Counter({0: 7559, 1: 877})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:15<00:00,  1.26it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.8172 (0.8036-0.8309),0.4334 (0.4087-0.4581),0.9068 (0.9014-0.9122),0.8886 (0.8814-0.8957),0.9068 (0.9014-0.9122),0.8843 (0.8766-0.8921),0.3368 (0.3065-0.367),0.0744 (0.0711-0.0777)
RF,0.8062 (0.7913-0.8211),0.4367 (0.4049-0.4685),0.9021 (0.8959-0.9083),0.8933 (0.8818-0.9047),0.9021 (0.8959-0.9083),0.8654 (0.8561-0.8747),0.2412 (0.1892-0.2931),0.0771 (0.0736-0.0805)
XGB,0.7929 (0.7753-0.8105),0.4044 (0.3777-0.4312),0.9041 (0.8995-0.9087),0.884 (0.877-0.8911),0.9041 (0.8995-0.9087),0.8844 (0.8786-0.8902),0.3359 (0.3022-0.3695),0.0788 (0.0754-0.0823)


In [50]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Urinary Disorders',
       'Streptococcal sepsis', 'First hosp stay', 'First careunit',
       'COPD ', 'Hosp counts', 'Type 2 Diabetes', 'Respiratory Disorders'],
      dtype=object)

### Based on 'gender' Group

In [51]:
print(Counter(all_df['Gender']))
IV_gender1 = all_df[all_df['Gender']>0]
IV_gender0 = all_df[all_df['Gender']==0]

Counter({1: 21394, 0: 15786})


In [52]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = IV_gender1.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 19465, 1: 1929}) (21394, 70) 9.0%
Train: Counter({0: 13605, 1: 1370})
Test: Counter({0: 5860, 1: 559})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:14<00:00,  1.35it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7982 (0.7813-0.815),0.3786 (0.3422-0.415),0.9178 (0.9138-0.9218),0.8991 (0.8924-0.9058),0.9178 (0.9138-0.9218),0.8986 (0.8935-0.9036),0.3139 (0.2711-0.3566),0.0682 (0.0646-0.0718)
RF,0.7921 (0.7733-0.811),0.3429 (0.3011-0.3848),0.9162 (0.9099-0.9224),0.9037 (0.8863-0.9211),0.9162 (0.9099-0.9224),0.8808 (0.8714-0.8903),0.1637 (0.1329-0.1946),0.0682 (0.0645-0.0718)
XGB,0.7782 (0.7598-0.7966),0.3276 (0.2876-0.3676),0.9159 (0.9102-0.9216),0.8923 (0.8826-0.902),0.9159 (0.9102-0.9216),0.8936 (0.8856-0.9017),0.2591 (0.2099-0.3082),0.0727 (0.0681-0.0773)


In [53]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Streptococcal sepsis',
       'First hosp stay', 'Hosp counts', 'Respiratory Disorders',
       'Urinary Disorders', 'Self ADL', 'Use of assistive devices',
       'First careunit'], dtype=object)

In [54]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = IV_gender0.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 14242, 1: 1544}) (15786, 70) 9.8%
Train: Counter({0: 9986, 1: 1064})
Test: Counter({0: 4256, 1: 480})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:13<00:00,  1.44it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7846 (0.7612-0.8079),0.3911 (0.347-0.4352),0.9042 (0.896-0.9124),0.8832 (0.8718-0.8946),0.9042 (0.896-0.9124),0.8805 (0.8697-0.8912),0.2949 (0.252-0.3377),0.0774 (0.0714-0.0834)
RF,0.7827 (0.7688-0.7965),0.3917 (0.3444-0.4391),0.9044 (0.8979-0.911),0.8971 (0.8811-0.913),0.9044 (0.8979-0.911),0.8667 (0.8588-0.8747),0.2134 (0.1624-0.2643),0.0767 (0.0722-0.0812)
XGB,0.7511 (0.7264-0.7757),0.3468 (0.3008-0.3928),0.9027 (0.8932-0.9122),0.8799 (0.8678-0.892),0.9027 (0.8932-0.9122),0.8765 (0.8634-0.8897),0.2838 (0.2393-0.3284),0.0836 (0.0751-0.0921)


In [55]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Streptococcal sepsis',
       'Urinary Disorders', 'COPD ', 'Type 2 Diabetes',
       'Invasive Ventilation', 'Direct Bilirubin', 'Pneumonia',
       'Self ADL'], dtype=object)

### Based on 'RACE' Group

In [56]:
print(Counter(all_df['RACE']))

Counter({'WHITE': 23654, 'UNKNOWN': 3862, 'BLACK/AFRICAN AMERICAN': 3128, 'OTHER': 1219, 'WHITE - OTHER EUROPEAN': 647, 'UNABLE TO OBTAIN': 549, 'HISPANIC/LATINO - PUERTO RICAN': 472, 'ASIAN': 424, 'HISPANIC OR LATINO': 365, 'ASIAN - CHINESE': 358, 'WHITE - RUSSIAN': 325, 'HISPANIC/LATINO - DOMINICAN': 279, 'BLACK/CAPE VERDEAN': 247, 'PATIENT DECLINED TO ANSWER': 242, 'BLACK/CARIBBEAN ISLAND': 202, 'ASIAN - SOUTH EAST ASIAN': 155, 'PORTUGUESE': 151, 'BLACK/AFRICAN': 137, 'ASIAN - ASIAN INDIAN': 101, 'WHITE - EASTERN EUROPEAN': 82, 'AMERICAN INDIAN/ALASKA NATIVE': 80, 'WHITE - BRAZILIAN': 66, 'HISPANIC/LATINO - GUATEMALAN': 61, 'HISPANIC/LATINO - SALVADORAN': 56, 'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 48, 'HISPANIC/LATINO - CUBAN': 39, 'MULTIPLE RACE/ETHNICITY': 38, 'HISPANIC/LATINO - MEXICAN': 36, 'HISPANIC/LATINO - COLUMBIAN': 36, 'ASIAN - KOREAN': 34, 'HISPANIC/LATINO - CENTRAL AMERICAN': 32, 'HISPANIC/LATINO - HONDURAN': 28, 'SOUTH AMERICAN': 27})


In [57]:
WHITE = all_df[all_df['RACE'].str.contains('WHITE', case=False, na=False)]
WHITE.shape

(24774, 70)

In [58]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = WHITE.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 22411, 1: 2363}) (24774, 70) 9.5%
Train: Counter({0: 15677, 1: 1664})
Test: Counter({0: 6734, 1: 699})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:15<00:00,  1.33it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.8009 (0.7865-0.8153),0.4166 (0.3857-0.4476),0.9154 (0.911-0.9197),0.8984 (0.891-0.9058),0.9154 (0.911-0.9197),0.8953 (0.8882-0.9024),0.3402 (0.2933-0.3872),0.0686 (0.0653-0.0718)
RF,0.8002 (0.7825-0.8179),0.3915 (0.3543-0.4287),0.9118 (0.9067-0.9169),0.9025 (0.8902-0.9148),0.9118 (0.9067-0.9169),0.8767 (0.8697-0.8837),0.2263 (0.1719-0.2808),0.0714 (0.068-0.0747)
XGB,0.7762 (0.7568-0.7955),0.3748 (0.3405-0.409),0.9141 (0.9085-0.9198),0.8945 (0.8857-0.9033),0.9141 (0.9085-0.9198),0.8937 (0.8854-0.9019),0.3205 (0.2792-0.3617),0.0727 (0.0687-0.0767)


In [59]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Streptococcal sepsis',
       'First hosp stay', 'Self ADL', 'Urinary Disorders', 'Hosp counts',
       'COPD ', 'First careunit', 'Use of assistive devices'],
      dtype=object)

In [60]:
ASIAN = all_df[all_df['RACE'].str.contains('ASIAN', case=False, na=False)]
ASIAN.shape

(1072, 70)

In [61]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = ASIAN.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 979, 1: 93}) (1072, 70) 8.7%
Train: Counter({0: 681, 1: 69})
Test: Counter({0: 298, 1: 24})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:08<00:00,  2.25it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.8046 (0.6943-0.9149),0.4594 (0.2638-0.655),0.9191 (0.8879-0.9503),0.9119 (0.8768-0.947),0.9191 (0.8879-0.9503),0.9141 (0.8798-0.9484),0.3452 (0.1306-0.5597),0.0623 (0.044-0.0806)
RF,0.7254 (0.6202-0.8307),0.2341 (0.0731-0.3951),0.9241 (0.9006-0.9477),0.8712 (0.8162-0.9261),0.9241 (0.9006-0.9477),0.8907 (0.8563-0.9251),0.1154 (-0.0264-0.2572),0.0654 (0.0494-0.0813)
XGB,0.6965 (0.5765-0.8165),0.1857 (0.0605-0.3108),0.913 (0.8849-0.9411),0.8832 (0.8375-0.9288),0.913 (0.8849-0.9411),0.8913 (0.8612-0.9214),0.1087 (-0.0527-0.2701),0.0748 (0.056-0.0936)


In [62]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'Norepinephrine', 'Invasive Ventilation',
       'Hosp counts', 'AB_ID_COUNT', 'Self ADL', 'Arterial Line',
       'Glucose (whole blood)', 'Total Bilirubin', 'Total PEEP Level'],
      dtype=object)

In [63]:
OTHER_race = all_df[~all_df['RACE'].str.contains('WHITE|ASIAN', case=False, na=False)]
OTHER_race.shape

(11334, 70)

In [64]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = OTHER_race.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 10317, 1: 1017}) (11334, 70) 9.0%
Train: Counter({0: 7234, 1: 699})
Test: Counter({0: 3083, 1: 318})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:13<00:00,  1.51it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7885 (0.7681-0.8088),0.3532 (0.301-0.4053),0.9106 (0.9038-0.9174),0.8889 (0.8783-0.8996),0.9106 (0.9038-0.9174),0.8879 (0.8801-0.8957),0.278 (0.2141-0.3419),0.0736 (0.0693-0.078)
RF,0.7563 (0.7282-0.7844),0.2843 (0.2338-0.3347),0.9045 (0.8951-0.9138),0.8797 (0.8422-0.9172),0.9045 (0.8951-0.9138),0.8624 (0.8484-0.8763),0.1071 (0.0314-0.1828),0.0778 (0.071-0.0845)
XGB,0.755 (0.722-0.788),0.3035 (0.2374-0.3695),0.9049 (0.8944-0.9154),0.8722 (0.8515-0.8929),0.9049 (0.8944-0.9154),0.8779 (0.8631-0.8928),0.2014 (0.1088-0.294),0.0811 (0.0721-0.0901)


In [65]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Urinary Disorders',
       'First hosp stay', 'Dialysis - CRRT', 'Use of assistive devices',
       'COPD ', 'Self ADL', 'Streptococcal sepsis', 'Dialysis Catheter'],
      dtype=object)

### Based on Age Group 8–64 岁：非老年人（Non-elderly adults）≥65 岁：老年人（Older adults / Elderly）

In [66]:
Non_elderly = all_df[all_df.AGE<65]
Non_elderly.shape

(16413, 70)

In [67]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = Non_elderly.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 15026, 1: 1387}) (16413, 70) 8.5%
Train: Counter({0: 10496, 1: 993})
Test: Counter({0: 4530, 1: 394})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:13<00:00,  1.46it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7734 (0.7531-0.7936),0.3405 (0.2878-0.3931),0.9245 (0.9181-0.9308),0.9069 (0.8973-0.9166),0.9245 (0.9181-0.9308),0.9086 (0.9-0.9171),0.316 (0.2649-0.367),0.0624 (0.0575-0.0674)
RF,0.7763 (0.7562-0.7964),0.3241 (0.2817-0.3664),0.922 (0.9158-0.9281),0.8998 (0.88-0.9195),0.922 (0.9158-0.9281),0.889 (0.8782-0.8999),0.1538 (0.0566-0.2511),0.065 (0.0611-0.0688)
XGB,0.7668 (0.7419-0.7917),0.3192 (0.2791-0.3592),0.9192 (0.9128-0.9255),0.8992 (0.8898-0.9085),0.9192 (0.9128-0.9255),0.9014 (0.8942-0.9086),0.28 (0.2384-0.3216),0.0698 (0.0649-0.0747)


In [68]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'First hosp stay', 'AB_ID_COUNT', 'Self ADL',
       'COPD ', 'Streptococcal sepsis', 'Urinary Disorders',
       'First careunit', 'Use of assistive devices',
       'Respiratory Disorders'], dtype=object)

In [69]:
Elderly = all_df[all_df.AGE>64]
Elderly.shape

(20767, 70)

In [70]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = Elderly.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 18681, 1: 2086}) (20767, 70) 10.0%
Train: Counter({0: 13070, 1: 1466})
Test: Counter({0: 5611, 1: 620})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:14<00:00,  1.41it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.8 (0.782-0.8179),0.4452 (0.4171-0.4733),0.9112 (0.9061-0.9162),0.8948 (0.886-0.9035),0.9112 (0.9061-0.9162),0.89 (0.8824-0.8977),0.3467 (0.298-0.3955),0.0724 (0.0691-0.0758)
RF,0.7954 (0.7816-0.8093),0.3979 (0.3556-0.4401),0.9056 (0.8998-0.9114),0.8972 (0.8822-0.9123),0.9056 (0.8998-0.9114),0.8668 (0.8566-0.877),0.1944 (0.1301-0.2586),0.0754 (0.0723-0.0785)
XGB,0.7787 (0.7573-0.8),0.3714 (0.3324-0.4104),0.906 (0.8987-0.9132),0.8835 (0.8701-0.8968),0.906 (0.8987-0.9132),0.8833 (0.8744-0.8922),0.2916 (0.2381-0.345),0.0797 (0.0746-0.0848)


In [71]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Streptococcal sepsis',
       'Urinary Disorders', 'COPD ', 'Hosp counts', 'Self ADL',
       'Respiratory Disorders', 'Invasive Ventilation', 'First hosp stay'],
      dtype=object)

### Based on Admission Source

In [72]:
Emergency = all_df[all_df['ADMISSION_TYPE'].str.contains('EMER', case=False, na=False)]
Emergency.shape

(20436, 70)

In [73]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = Emergency.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 18272, 1: 2164}) (20436, 70) 10.6%
Train: Counter({0: 12822, 1: 1483})
Test: Counter({0: 5450, 1: 681})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:14<00:00,  1.40it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7596 (0.7409-0.7782),0.383 (0.3523-0.4137),0.8972 (0.8905-0.9039),0.8759 (0.8662-0.8856),0.8972 (0.8905-0.9039),0.8697 (0.861-0.8784),0.2929 (0.254-0.3318),0.0834 (0.0789-0.0879)
RF,0.7615 (0.7392-0.7839),0.3634 (0.317-0.4099),0.8931 (0.8861-0.9001),0.8803 (0.8647-0.8959),0.8931 (0.8861-0.9001),0.8491 (0.8376-0.8606),0.1673 (0.1133-0.2214),0.0855 (0.0813-0.0897)
XGB,0.7391 (0.7153-0.7628),0.3489 (0.3065-0.3913),0.8944 (0.8861-0.9027),0.8721 (0.8607-0.8836),0.8944 (0.8861-0.9027),0.868 (0.8574-0.8786),0.2828 (0.2434-0.3222),0.0905 (0.0839-0.0972)


In [74]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'Streptococcal sepsis', 'AB_ID_COUNT',
       'Urinary Disorders', 'Self ADL', 'COPD ', 'First hosp stay',
       'Use of assistive devices', 'Hosp counts', 'Type 2 Diabetes'],
      dtype=object)

In [75]:
URGENT = all_df[all_df['ADMISSION_TYPE'].str.contains('URGENT', case=False, na=False)]
URGENT.shape

(7516, 70)

In [76]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = URGENT.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 6743, 1: 773}) (7516, 70) 10.3%
Train: Counter({0: 4712, 1: 549})
Test: Counter({0: 2031, 1: 224})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:12<00:00,  1.63it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.8014 (0.7702-0.8325),0.4467 (0.384-0.5095),0.9112 (0.8979-0.9246),0.8951 (0.8766-0.9136),0.9112 (0.8979-0.9246),0.8907 (0.8722-0.9092),0.352 (0.2709-0.4331),0.0729 (0.0624-0.0834)
RF,0.7914 (0.7602-0.8226),0.4115 (0.3534-0.4697),0.8995 (0.8908-0.9082),0.8868 (0.8595-0.9142),0.8995 (0.8908-0.9082),0.8587 (0.8435-0.8739),0.1581 (0.0993-0.2168),0.0791 (0.0727-0.0855)
XGB,0.7647 (0.7379-0.7914),0.3289 (0.2556-0.4021),0.9066 (0.898-0.9153),0.8821 (0.8656-0.8986),0.9066 (0.898-0.9153),0.8821 (0.8664-0.8977),0.2614 (0.1726-0.3502),0.0814 (0.0735-0.0892)


In [77]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Respiratory Disorders', 'COPD ',
       'Urinary Disorders', 'Hosp counts', 'First hosp stay',
       'Streptococcal sepsis', 'Use of assistive devices', 'CKD'],
      dtype=object)

In [78]:
OBSERVATION = all_df[all_df['ADMISSION_TYPE'].str.contains('OBSERVATION', case=False, na=False)]
OBSERVATION.shape

(3753, 70)

In [79]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = OBSERVATION.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 3431, 1: 322}) (3753, 70) 8.6%
Train: Counter({0: 2409, 1: 218})
Test: Counter({0: 1022, 1: 104})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:10<00:00,  1.84it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7686 (0.7148-0.8223),0.3462 (0.2349-0.4575),0.9164 (0.902-0.9308),0.8917 (0.8676-0.9158),0.9164 (0.902-0.9308),0.8957 (0.8728-0.9186),0.2567 (0.1403-0.3731),0.0693 (0.0579-0.0807)
RF,0.7162 (0.6789-0.7535),0.2615 (0.1849-0.338),0.9112 (0.8943-0.9281),0.8638 (0.8059-0.9217),0.9112 (0.8943-0.9281),0.8721 (0.849-0.8952),0.0828 (-0.0128-0.1784),0.0751 (0.0633-0.0869)
XGB,0.7043 (0.6523-0.7564),0.2152 (0.1459-0.2845),0.9055 (0.8899-0.9211),0.8673 (0.8435-0.8912),0.9055 (0.8899-0.9211),0.8765 (0.8575-0.8954),0.1173 (0.0344-0.2003),0.0852 (0.0717-0.0987)


In [80]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Temperature', 'Hosp counts',
       'Use of assistive devices', 'First hosp stay', 'Direct Bilirubin',
       'WBC', 'Total Bilirubin', 'Glucose (whole blood)'], dtype=object)

In [81]:
ELECTIVE = all_df[all_df['ADMISSION_TYPE'].str.contains('ELECTIVE', case=False, na=False)]
ELECTIVE.shape

(1810, 70)

In [82]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = ELECTIVE.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 1741, 1: 69}) (1810, 70) 3.8%
Train: Counter({0: 1215, 1: 52})
Test: Counter({0: 526, 1: 17})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:08<00:00,  2.24it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.8211 (0.691-0.9511),0.3777 (0.1731-0.5823),0.9658 (0.9482-0.9835),0.968 (0.9532-0.9827),0.9658 (0.9482-0.9835),0.9669 (0.9513-0.9826),0.4441 (0.2224-0.6658),0.0272 (0.0184-0.036)
RF,0.9257 (0.8537-0.9976),0.7069 (0.4485-0.9653),0.9843 (0.9722-0.9963),0.978 (0.9596-0.9963),0.9843 (0.9722-0.9963),0.9781 (0.9601-0.9962),0.5425 (0.156-0.9291),0.0176 (0.0101-0.0251)
XGB,0.9069 (0.8398-0.974),0.5856 (0.3764-0.7948),0.9816 (0.9742-0.9891),0.9793 (0.9701-0.9886),0.9816 (0.9742-0.9891),0.9771 (0.9662-0.988),0.5644 (0.3724-0.7565),0.0183 (0.0121-0.0245)


In [83]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'Total Bilirubin', 'Heart Rate', 'WBC',
       'Calcium non-ionized', 'Arterial Line', 'First careunit',
       'AB_ID_COUNT', 'Norepinephrine', 'Prothrombin time'], dtype=object)

In [84]:
MICU_SICU = all_df[all_df['FIRST_CAREUNIT'].isin(['Medical Intensive Care Unit (MICU)',
                                                 'Surgical Intensive Care Unit (SICU)',
                                                 'Medical/Surgical Intensive Care Unit (MICU/SICU)'])]
MICU = all_df[all_df['FIRST_CAREUNIT'] == 'Medical Intensive Care Unit (MICU)']
SICU = all_df[all_df['FIRST_CAREUNIT'] == 'Surgical Intensive Care Unit (SICU)']
CCU = all_df[all_df['FIRST_CAREUNIT'] == 'Coronary Care Unit (CCU)']
TSICU = all_df[all_df['FIRST_CAREUNIT'] == 'Trauma SICU (TSICU)']
print(MICU_SICU.shape, MICU.shape, SICU.shape, CCU.shape, TSICU.shape)

(19775, 70) (8375, 70) (5757, 70) (4060, 70) (4184, 70)


In [85]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = MICU_SICU.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 17372, 1: 2403}) (19775, 70) 12.2%
Train: Counter({0: 12151, 1: 1691})
Test: Counter({0: 5221, 1: 712})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:15<00:00,  1.27it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7929 (0.7764-0.8094),0.4349 (0.399-0.4708),0.8909 (0.8851-0.8967),0.871 (0.8615-0.8805),0.8909 (0.8851-0.8967),0.8636 (0.8548-0.8723),0.3259 (0.2839-0.3679),0.0866 (0.0824-0.0909)
RF,0.7799 (0.7629-0.7968),0.4283 (0.3879-0.4687),0.89 (0.8841-0.8959),0.8825 (0.8686-0.8963),0.89 (0.8841-0.8959),0.8472 (0.8381-0.8564),0.2429 (0.1976-0.2883),0.0862 (0.0829-0.0894)
XGB,0.7532 (0.7261-0.7804),0.3893 (0.354-0.4246),0.8882 (0.8822-0.8942),0.8649 (0.856-0.8738),0.8882 (0.8822-0.8942),0.8634 (0.854-0.8727),0.3094 (0.2705-0.3484),0.0937 (0.0886-0.0989)


In [86]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'First hosp stay', 'COPD ',
       'Streptococcal sepsis', 'Self ADL', 'Urinary Disorders',
       'Respiratory Disorders', 'Hosp counts', 'Dialysis Catheter'],
      dtype=object)

In [87]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = MICU.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 7210, 1: 1165}) (8375, 70) 13.9%
Train: Counter({0: 5036, 1: 826})
Test: Counter({0: 2174, 1: 339})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:13<00:00,  1.51it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7507 (0.7076-0.7938),0.4338 (0.3645-0.5032),0.8824 (0.8678-0.897),0.8633 (0.8407-0.886),0.8824 (0.8678-0.897),0.8542 (0.8336-0.8749),0.3432 (0.2703-0.4161),0.0956 (0.0848-0.1063)
RF,0.7607 (0.7304-0.7911),0.4213 (0.3648-0.4778),0.8754 (0.865-0.8858),0.861 (0.8348-0.8873),0.8754 (0.865-0.8858),0.8294 (0.8105-0.8483),0.2126 (0.1465-0.2786),0.0988 (0.0927-0.1049)
XGB,0.7293 (0.7029-0.7557),0.3746 (0.3233-0.4258),0.8745 (0.8599-0.889),0.8489 (0.8298-0.8679),0.8745 (0.8599-0.889),0.849 (0.8328-0.8653),0.2859 (0.228-0.3438),0.1066 (0.0948-0.1184)


In [88]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'First hosp stay', 'AB_ID_COUNT',
       'Urinary Disorders', 'COPD ', 'Hosp counts', 'Self ADL',
       'Streptococcal sepsis', 'CKD', 'Use of assistive devices'],
      dtype=object)

In [89]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = SICU.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 5179, 1: 578}) (5757, 70) 10.0%
Train: Counter({0: 3622, 1: 407})
Test: Counter({0: 1557, 1: 171})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:12<00:00,  1.66it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7587 (0.7183-0.7992),0.3783 (0.3187-0.4378),0.9118 (0.9016-0.9219),0.8914 (0.8756-0.9073),0.9118 (0.9016-0.9219),0.8883 (0.8736-0.9031),0.2933 (0.2208-0.3658),0.0732 (0.0651-0.0814)
RF,0.733 (0.6841-0.7819),0.3355 (0.2684-0.4025),0.9012 (0.8851-0.9173),0.8713 (0.8256-0.917),0.9012 (0.8851-0.9173),0.8633 (0.8366-0.8899),0.1769 (0.0267-0.3271),0.0801 (0.0699-0.0903)
XGB,0.7069 (0.6666-0.7471),0.2957 (0.2342-0.3572),0.9031 (0.8929-0.9132),0.875 (0.8532-0.8968),0.9031 (0.8929-0.9132),0.8758 (0.8608-0.8908),0.232 (0.1217-0.3423),0.0869 (0.0777-0.0962)


In [90]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Streptococcal sepsis', 'CKD',
       'Use of assistive devices', 'COPD ', 'Hosp counts',
       'Urinary Disorders', 'Dialysis Catheter', 'Direct Bilirubin'],
      dtype=object)

In [91]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = CCU.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 3770, 1: 290}) (4060, 70) 7.1%
Train: Counter({0: 2636, 1: 206})
Test: Counter({0: 1134, 1: 84})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:11<00:00,  1.77it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7165 (0.652-0.781),0.3092 (0.2137-0.4048),0.9331 (0.9194-0.9469),0.9151 (0.8965-0.9337),0.9331 (0.9194-0.9469),0.9149 (0.8966-0.9333),0.2802 (0.1911-0.3694),0.058 (0.0477-0.0683)
RF,0.7112 (0.6493-0.7732),0.2033 (0.1235-0.2831),0.9306 (0.9187-0.9425),0.9002 (0.8559-0.9445),0.9306 (0.9187-0.9425),0.8993 (0.8825-0.9162),0.1056 (-0.0062-0.2174),0.062 (0.0532-0.0707)
XGB,0.6483 (0.5827-0.7138),0.1471 (0.0928-0.2015),0.9294 (0.917-0.9418),0.8979 (0.8557-0.9401),0.9294 (0.917-0.9418),0.9021 (0.8824-0.9218),0.1183 (-0.0235-0.2601),0.0649 (0.0526-0.0771)


In [92]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Self ADL', 'Glucose (whole blood)',
       'Hosp counts', 'COPD ', 'Use of assistive devices',
       'First hosp stay', 'Urinary Disorders', 'WBC'], dtype=object)

In [93]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = TSICU.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 3789, 1: 395}) (4184, 70) 9.4%
Train: Counter({0: 2650, 1: 278})
Test: Counter({0: 1139, 1: 117})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:11<00:00,  1.76it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7697 (0.7168-0.8226),0.3994 (0.3043-0.4945),0.9136 (0.9013-0.926),0.8977 (0.8807-0.9147),0.9136 (0.9013-0.926),0.8905 (0.8744-0.9067),0.3116 (0.2306-0.3926),0.071 (0.0611-0.0809)
RF,0.7453 (0.7129-0.7777),0.323 (0.2589-0.387),0.9124 (0.897-0.9277),0.8874 (0.8418-0.933),0.9124 (0.897-0.9277),0.8762 (0.855-0.8974),0.1772 (0.0584-0.296),0.0729 (0.0641-0.0817)
XGB,0.678 (0.6386-0.7174),0.239 (0.173-0.305),0.9077 (0.8965-0.9189),0.8785 (0.8535-0.9036),0.9077 (0.8965-0.9189),0.8765 (0.8574-0.8957),0.1743 (0.07-0.2785),0.0834 (0.0722-0.0946)


In [94]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Urinary Disorders',
       'Use of assistive devices', 'WBC', 'Norepinephrine', 'Hosp counts',
       'First hosp stay', 'Streptococcal sepsis', 'Total Bilirubin'],
      dtype=object)

### 'Use of assistive devices'

In [108]:
All_40815_df_nan = pd.read_csv('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/All_40815_df_nan.csv')
All_40815_df_nan = All_40815_df_nan.rename(columns=id_mapping)
All_40815_df_nan = All_40815_df_nan[All_40815_df_nan.ICUSTAY_ID.isin(all_IV.ICUSTAY_ID)]
All_40815_df_nan['Use of assistive devices'].value_counts()

None          9379
Walker        1979
Cane          1518
Wheelchair    1145
0              514
Other          395
Name: Use of assistive devices, dtype: int64

In [112]:
all_df['Use of assistive devices'].value_counts()

2    31629
4     1979
1     1518
5     1145
0      514
3      395
Name: Use of assistive devices, dtype: int64

In [167]:
Use_assistive_devices = all_df[all_df['Use of assistive devices'].isin([1,4,5,3])]
Not_Use_assistive_devices = all_df[~(all_df['Use of assistive devices'].isin([1,4,5,3]))]
print(Use_assistive_devices.shape, Not_Use_assistive_devices.shape)

(5037, 71) (32143, 71)


In [168]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = Use_assistive_devices.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 4392, 1: 645}) (5037, 71) 12.8%
Train: Counter({0: 3071, 1: 454})
Test: Counter({0: 1321, 1: 191})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:12<00:00,  1.66it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7457 (0.7009-0.7905),0.3883 (0.3218-0.4547),0.8829 (0.8676-0.8982),0.8575 (0.8346-0.8804),0.8829 (0.8676-0.8982),0.8588 (0.8395-0.8781),0.2906 (0.2138-0.3675),0.0934 (0.0825-0.1042)
RF,0.7172 (0.6731-0.7612),0.3366 (0.2767-0.3964),0.8757 (0.8604-0.8909),0.8374 (0.776-0.8988),0.8757 (0.8604-0.8909),0.8248 (0.8013-0.8482),0.1173 (-0.0071-0.2418),0.0984 (0.0896-0.1073)
XGB,0.6904 (0.6585-0.7222),0.2731 (0.2145-0.3316),0.871 (0.8556-0.8863),0.8242 (0.7961-0.8522),0.871 (0.8556-0.8863),0.8315 (0.8125-0.8504),0.1507 (0.0746-0.2267),0.1107 (0.0991-0.1223)


In [169]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'AB_ID_COUNT', 'Dialysis Catheter',
       'Invasive Ventilation', 'Streptococcal sepsis', 'Self ADL',
       'Urinary Disorders', 'Respiratory Disorders', 'Type 2 Diabetes',
       'Hosp counts'], dtype=object)

In [111]:
Walker = all_df[all_df['Use of assistive devices'] == 4]
Wheelchair = all_df[all_df['Use of assistive devices'] == 5]
Cane = all_df[all_df['Use of assistive devices'] == 1]
print(Walker.shape,Wheelchair.shape,Cane.shape)

(1979, 70) (1145, 70) (1518, 70)


In [113]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = Walker.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 1747, 1: 232}) (1979, 70) 11.7%
Train: Counter({0: 1218, 1: 167})
Test: Counter({0: 529, 1: 65})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:09<00:00,  2.11it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.7141 (0.6152-0.813),0.363 (0.1966-0.5294),0.9005 (0.8785-0.9226),0.8856 (0.8491-0.922),0.9005 (0.8785-0.9226),0.8733 (0.8398-0.9068),0.3134 (0.1316-0.4952),0.0877 (0.0718-0.1035)
RF,0.6775 (0.6095-0.7454),0.216 (0.1287-0.3033),0.8983 (0.877-0.9195),0.841 (0.778-0.9041),0.8983 (0.877-0.9195),0.8567 (0.8243-0.8891),0.1058 (-0.0207-0.2323),0.09 (0.0763-0.1037)
XGB,0.6317 (0.5171-0.7463),0.2153 (0.1252-0.3054),0.8763 (0.8519-0.9008),0.8196 (0.7656-0.8736),0.8763 (0.8519-0.9008),0.8405 (0.8041-0.8769),0.1152 (-0.0548-0.2853),0.1086 (0.0866-0.1307)


In [114]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'Glucose (whole blood)', 'AB_ID_COUNT',
       'Streptococcal sepsis', 'WBC', 'Arterial Line',
       'Invasive Ventilation', 'Total PEEP Level', 'Pneumonia',
       'Arterial CO2 Pressure'], dtype=object)

In [115]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = Wheelchair.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 936, 1: 209}) (1145, 70) 18.3%
Train: Counter({0: 659, 1: 142})
Test: Counter({0: 277, 1: 67})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:09<00:00,  2.08it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.6927 (0.6157-0.7698),0.4138 (0.2734-0.5542),0.8182 (0.776-0.8605),0.7822 (0.7217-0.8427),0.8182 (0.776-0.8605),0.7821 (0.7304-0.8338),0.2576 (0.107-0.4083),0.1404 (0.1116-0.1693)
RF,0.6581 (0.5682-0.7481),0.3644 (0.261-0.4678),0.811 (0.7818-0.8403),0.7432 (0.6204-0.8661),0.811 (0.7818-0.8403),0.7333 (0.6891-0.7775),0.1229 (-0.0272-0.2731),0.149 (0.1265-0.1714)
XGB,0.6624 (0.6068-0.718),0.3645 (0.2435-0.4854),0.795 (0.7613-0.8286),0.7511 (0.6744-0.8278),0.795 (0.7613-0.8286),0.7411 (0.6977-0.7844),0.1299 (-0.0092-0.269),0.1742 (0.1464-0.2021)


In [116]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'Invasive Ventilation', 'Glucose (whole blood)',
       'WBC', 'Total PEEP Level', 'COPD ', 'Streptococcal sepsis',
       'Arterial Line', 'Creatinine (serum)', 'Platelet Count'],
      dtype=object)

In [117]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = Cane.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 1391, 1: 127}) (1518, 70) 8.4%
Train: Counter({0: 969, 1: 93})
Test: Counter({0: 422, 1: 34})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:09<00:00,  2.18it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.6923 (0.528-0.8567),0.2899 (0.086-0.4938),0.9146 (0.877-0.9522),0.8889 (0.8331-0.9446),0.9146 (0.877-0.9522),0.898 (0.854-0.9421),0.1893 (-0.0453-0.4238),0.0715 (0.0431-0.0999)
RF,0.7044 (0.6113-0.7974),0.2278 (0.0991-0.3566),0.9265 (0.91-0.943),0.8677 (0.8281-0.9073),0.9265 (0.91-0.943),0.8912 (0.8671-0.9153),0.0562 (0.0-0.1124),0.0662 (0.0544-0.078)
XGB,0.6917 (0.6141-0.7694),0.2275 (0.0968-0.3582),0.9144 (0.8945-0.9342),0.8811 (0.8308-0.9313),0.9144 (0.8945-0.9342),0.8897 (0.8625-0.9168),0.1553 (-0.0377-0.3482),0.0765 (0.0603-0.0926)


In [118]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['MDR_BEFORE', 'Hosp counts', 'AB_ID_COUNT', 'Streptococcal sepsis',
       'Glucose (whole blood)', 'First careunit', 'Total PEEP Level',
       'Norepinephrine', 'WBC', 'Alkaline Phosphate'], dtype=object)

### 'Invasive Ventilation'

In [160]:
InVents = all_df[all_df.InvasiveVent == 1]
InVents.shape

(7752, 71)

In [161]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = InVents.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
# # SMOTE
# smote = SMOTE(random_state=42)
# smote_x, smote_Y = smote.fit_resample(Train[features], Train[['MDR_label']])
# Train =  pd.concat([smote_x, smote_Y],axis=1)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
# dill.dump(all_AUROC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUROC.pkl', 'wb'))
# dill.dump(all_AUPRC, open('D:/2024BI_Journal/MIMIC_IV/Median_B/Final/AUROC/MDR_AUPRC.pkl', 'wb'))
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)

60
Counter({0: 7278, 1: 474}) (7752, 71) 6.1%
Train: Counter({0: 5099, 1: 327})
Test: Counter({0: 2179, 1: 147})


Model: xgboost: 100%|██████████████████████████████████████████████████████████████████| 20/20 [00:11<00:00,  1.71it/s]


,ROC_AUC,PRC_AUC,Accuracy,Precision,Recall,F1,MCC,Brier
LR,0.8861 (0.8637-0.9085),0.4733 (0.3882-0.5584),0.9458 (0.9393-0.9523),0.936 (0.9264-0.9456),0.9458 (0.9393-0.9523),0.9375 (0.9278-0.9471),0.4386 (0.3556-0.5217),0.0429 (0.0381-0.0476)
RF,0.8894 (0.8644-0.9144),0.4945 (0.4127-0.5762),0.9371 (0.9293-0.945),0.9276 (0.9072-0.9481),0.9371 (0.9293-0.945),0.9136 (0.9015-0.9258),0.2448 (0.1295-0.3602),0.0465 (0.0405-0.0525)
XGB,0.8794 (0.8543-0.9045),0.4424 (0.359-0.5259),0.9426 (0.9354-0.9497),0.9302 (0.9189-0.9416),0.9426 (0.9354-0.9497),0.9285 (0.9182-0.9389),0.3482 (0.26-0.4364),0.0475 (0.0413-0.0537)


In [162]:
all_importances[~all_importances.Feature.isin(cultures)].sort_values(by='xgboost', ascending=False).head(10).Feature.values

array(['First careunit', 'MDR_BEFORE', 'Invasive Ventilation',
       'AB_ID_COUNT', 'Respiratory Disorders', 'Pneumonia', 'WBC',
       'Use of assistive devices', 'Arterial Line', 'Self ADL'],
      dtype=object)